## Train many models together to compare which works best


In [46]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# sklearn packages
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score



In [2]:
# iris is a multi variate dataset.
df = sns.load_dataset('iris')
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [3]:
X = df.drop('species', axis=1)
Y = df['species']

In [4]:
Y.unique()

array(['setosa', 'versicolor', 'virginica'], dtype=object)

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

In [6]:
model_svm = SVC(C=5.0, kernel='rbf', gamma = 'auto', random_state=42)
model_svm.fit(X_train, Y_train)

SVC(C=5.0, gamma='auto', random_state=42)

In [7]:
y_pred_svm = model_svm.predict(X_test)

In [8]:
accuracy_svm = model_svm.score(X_test, Y_test)
print("Accuracy of SVM model:", accuracy_svm)

# there's a 100% accuracy. It means it is overfitting the data.

Accuracy of SVM model: 1.0


In [9]:
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train, Y_train)

KNeighborsClassifier()

In [10]:
y_pred_knn = model_knn.predict(X_test)

In [11]:
accuracy_knn = model_knn.score(X_test, Y_test)
print("Accuracy of KNN model:", accuracy_knn)

Accuracy of KNN model: 1.0


### Now we will use Grid Search Cross Validation to find the best set of hyperparameters to train the model on the data.

In [12]:

from sklearn.model_selection import GridSearchCV

classifier = GridSearchCV((model_svm), {
    'C': [1, 3, 5, 7, 10, 16, 17],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf', 'poly']
}, cv=7, return_train_score=False)

In [13]:
# we don't need to do train test slit for Grid Search CV, so we will use X and Y directly.

classifier.fit(X, Y)

GridSearchCV(cv=7, estimator=SVC(C=5.0, gamma='auto', random_state=42),
             param_grid={'C': [1, 3, 5, 7, 10, 16, 17],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'poly']})

In [14]:
results = pd.DataFrame(classifier.cv_results_)
# results[['param_C', 'param_gamma', 'param_kernel', 'mean_test_score']]

### tuning the hyperparamaters for knn model

In [15]:
classifier1 = GridSearchCV((model_knn), {
    'n_neighbors': [1, 3, 5, 7, 10, 15, 20],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}, cv=7, return_train_score=False)

In [16]:
classifier1.fit(X, Y)

GridSearchCV(cv=7, estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': [1, 3, 5, 7, 10, 15, 20],
                         'weights': ['uniform', 'distance']})

In [17]:
results1 = pd.DataFrame(classifier1.cv_results_)
# results1[['param_n_neighbors', 'param_weights', 'param_algorithm', 'mean_test_score']]

### using random search CV for both SVM and KNN models

In [18]:
from sklearn.model_selection import RandomizedSearchCV

In [19]:
classifier_r = RandomizedSearchCV((model_svm), {
    'C': [1, 3, 5, 7, 10, 16, 17],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf', 'poly']
}, n_iter = 6, cv=7, return_train_score=False)

In [20]:
classifier_r.fit(X,Y)

RandomizedSearchCV(cv=7, estimator=SVC(C=5.0, gamma='auto', random_state=42),
                   n_iter=6,
                   param_distributions={'C': [1, 3, 5, 7, 10, 16, 17],
                                        'gamma': ['scale', 'auto'],
                                        'kernel': ['linear', 'rbf', 'poly']})

In [21]:
results_r = pd.DataFrame(classifier_r.cv_results_)
results_r[['param_C', 'param_gamma', 'param_kernel', 'mean_test_score']]

,param_C,param_gamma,param_kernel,mean_test_score
0,7,scale,linear,0.973098
1,17,auto,rbf,0.973408
2,1,scale,poly,0.966605
3,5,scale,rbf,0.973408
4,16,scale,poly,0.959493
5,10,auto,rbf,0.980210


In [22]:
# for KNN model

classifier_rknn = RandomizedSearchCV((model_knn), {
    'n_neighbors': [1, 3, 5, 7, 10, 15, 20],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}, n_iter = 16, cv=7, return_train_score=False)

In [23]:
classifier_rknn.fit(X,Y)

RandomizedSearchCV(cv=7, estimator=KNeighborsClassifier(), n_iter=16,
                   param_distributions={'algorithm': ['auto', 'ball_tree',
                                                      'kd_tree', 'brute'],
                                        'n_neighbors': [1, 3, 5, 7, 10, 15, 20],
                                        'weights': ['uniform', 'distance']})

In [24]:
results_rknn = pd.DataFrame(classifier_rknn.cv_results_)
results_rknn[['param_n_neighbors', 'param_weights', 'param_algorithm', 'mean_test_score', 'rank_test_score']]

,param_n_neighbors,param_weights,param_algorithm,mean_test_score,rank_test_score
0,1,distance,ball_tree,0.960111,9
1,1,uniform,kd_tree,0.960111,9
2,20,uniform,auto,0.959802,14
3,1,uniform,auto,0.960111,9
4,5,distance,auto,0.966605,4
5,1,uniform,brute,0.960111,9
6,10,uniform,auto,0.966296,8
7,10,distance,ball_tree,0.980210,1
8,3,distance,ball_tree,0.966914,3
9,15,uniform,brute,0.959802,14


## Ensemble learning
### types: 
###        (1) stacking
###        (2) Bagging: RandomForest 
###        (3) Bossting: AdaBoost, XGBoost, Gradient boosting
training different models together on same data, use their results as inputs to train the final model depending on the type of ensemble learning.


In [28]:
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()


In [32]:
# we have X (features) and Y (target variable). We need to encode the target variable Y into numerical values for model training.
Y_le = le.fit_transform(Y)

In [33]:
# we need to train test split the data again because we have changed the target variable Y into numerical values.
X_train_en, X_test_en, Y_train_en, Y_test_en = train_test_split(X, Y_le, test_size=0.20, random_state=42, stratify = Y_le)

In [36]:
# define the base learners for the stacking classifier. (Decision Tree, SVC, and Logistic regression). Meta learner would be KNN, Logistioc Regression and SVM.
# In future, we will use Gradient Boosting and Random Forest as base learners as well. We will also use XGBoost as base learners.
base_learners = [
    ('svm', SVC(C=5.0, kernel='rbf', gamma = 'auto', random_state=42, probability=True)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('lr', LogisticRegression(max_iter =1000, random_state=42))
]

In [37]:
# meta learners
meta_learner_1 = KNeighborsClassifier(n_neighbors=5)
meta_learner_2 = LogisticRegression(max_iter=1000, random_state=42)
meta_learner_3 = SVC(C=5.0, kernel='rbf', gamma='auto', random_state=42, probability=True)

In [38]:
# stacking classifiers with different meta learners 
stacking_clf_1 = StackingClassifier(estimators=base_learners, final_estimator=meta_learner_1, cv=7, n_jobs=-1)

stacking_clf_2 = StackingClassifier(estimators=base_learners, final_estimator=meta_learner_2, cv=7, n_jobs=-1)

stacking_clf_3 = StackingClassifier(estimators=base_learners, final_estimator=meta_learner_3, cv=7, n_jobs=-1)

In [40]:
# fitting with stacking classifier 1
stacking_clf_1.fit(X_train_en, Y_train_en)

StackingClassifier(cv=7,
                   estimators=[('svm',
                                SVC(C=5.0, gamma='auto', probability=True,
                                    random_state=42)),
                               ('knn', KNeighborsClassifier()),
                               ('dt',
                                DecisionTreeClassifier(max_depth=3,
                                                       random_state=42)),
                               ('lr',
                                LogisticRegression(max_iter=1000,
                                                   random_state=42))],
                   final_estimator=KNeighborsClassifier(), n_jobs=-1)

In [41]:
# fitting with stacking classifier 2
stacking_clf_2.fit(X_train_en, Y_train_en)

StackingClassifier(cv=7,
                   estimators=[('svm',
                                SVC(C=5.0, gamma='auto', probability=True,
                                    random_state=42)),
                               ('knn', KNeighborsClassifier()),
                               ('dt',
                                DecisionTreeClassifier(max_depth=3,
                                                       random_state=42)),
                               ('lr',
                                LogisticRegression(max_iter=1000,
                                                   random_state=42))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      random_state=42),
                   n_jobs=-1)

In [42]:
# fitting with stacking classifier 3
stacking_clf_3.fit(X_train_en, Y_train_en)

StackingClassifier(cv=7,
                   estimators=[('svm',
                                SVC(C=5.0, gamma='auto', probability=True,
                                    random_state=42)),
                               ('knn', KNeighborsClassifier()),
                               ('dt',
                                DecisionTreeClassifier(max_depth=3,
                                                       random_state=42)),
                               ('lr',
                                LogisticRegression(max_iter=1000,
                                                   random_state=42))],
                   final_estimator=SVC(C=5.0, gamma='auto', probability=True,
                                       random_state=42),
                   n_jobs=-1)

In [43]:
# doing the predictions with stacking classifier 1
y_pred_stack_1 = stacking_clf_1.predict(X_test_en)

In [44]:
# prediction with stacking classifier 2
y_pred_stack_2 = stacking_clf_2.predict(X_test_en)

In [45]:
# predictions with stacking classifier 3
y_pred_stack_3 = stacking_clf_3.predict(X_test_en)

In [47]:
# finding out the accuracy score of the stacking classifiers
accuracy_stack_1 = accuracy_score(y_pred_stack_1, Y_test_en)

accuracy_stack_2 = accuracy_score(y_pred_stack_2, Y_test_en)

accuracy_stack_3 = accuracy_score(y_pred_stack_3, Y_test_en)

In [48]:
accuracy_stack_1, accuracy_stack_2, accuracy_stack_3

(0.9666666666666667, 0.9666666666666667, 0.9666666666666667)

In [49]:
y_pred_stack_1, y_pred_stack_2, y_pred_stack_3

(array([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2,
        1, 2, 2, 2, 1, 0, 2, 0]),
 array([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2,
        1, 2, 2, 2, 1, 0, 2, 0]),
 array([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2,
        1, 2, 2, 2, 1, 0, 2, 0]))

Exception ignored in: <function ResourceTracker.__del__ at 0x788cbbf82660>
Traceback (most recent call last):
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x740e26d86660>
Traceback (most recent call last):
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/anukul/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__